In [ ]:
from pathlib import Path
from metasmith.agents import Agent
from metasmith.models.libraries import *
from metasmith.models.remote import *

from local.constants import WORKSPACE_ROOT
SOCKEYE_SOURCE = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=64a5c402-05c4-4607-bbad-46a9c2aebd98&origin_path=%2Fhome%2Ftxyliu%2F")
SOCKEYE_SOURCE.endpoint

In [ ]:
agent_local = Agent(
    home=Source.FromLocal(WORKSPACE_ROOT/"main/local_mock/cache/local_home"),
)

agent_ssh = Agent(
    home=SshSource(
        host="cosmos",
        path="~/workspace/metasmith_home",
    ).AsSource(),
)

agent_slurm = Agent(
    setup_commands=[
        "module load gcc/9.4.0 apptainer/1.3.1",
    ],
    home=SshSource(
        host="sockeye",
        path="~/scratch/metasmith_home",
    ).AsSource(),
    globus_uuid=SOCKEYE_SOURCE.endpoint,
)

agent=agent_local
# agent=agent_ssh
# agent=agent_slurm
agent.Deploy()

In [ ]:
CACHE = WORKSPACE_ROOT/"main/local_mock/cache/xgdb_tests"
trlib = TransformInstanceLibrary.Load("./transforms/simple_1")
xgdb = DataInstanceLibrary.Load(CACHE/"test.xgdb")
# refdb = DataInstanceLibrary.Load(CACHE/"ref.xgdb")
types = DataTypeLibrary.Load(WORKSPACE_ROOT/"main/local_mock/prototypes/metagenomics.dev3.yml")

In [ ]:
chinook_ep = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2F").endpoint
# refdb.SaveAs(GlobusSource(endpoint=chinook_ep, path="/Metasmith/ref.xgdb").AsSource())

refdb = DataInstanceLibrary.LoadFrom(
    src=GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fref.xgdb%2F").AsSource(),
    dest=CACHE/"ref.image.xgdb",
    as_image=True,
)
refdb.remote_src

In [ ]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[trlib],
    targets=[
        types["orf_annotations"].WithLineage([
            types["contigs"],
            # xgdb["example.fna"].type,
        ]),
    ],
)

print(task.plan._key)
for step in task.plan.steps:
    print(step.transform.name)

In [ ]:
with open(WORKSPACE_ROOT/"secrets/slurm_account") as f:
    slurm_account = f.read().strip()
    
task.container_runtime = Runtime.APPTAINER
task.config = dict(
    nextflow = dict(
        preset = "default",
        # preset = "slurm",
        slurm_account=slurm_account,
    ),

)

In [ ]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

In [ ]:
# import shutil
# work_root = WORKSPACE_ROOT/"main/local_mock/cache/local_home/runs/kCvaS6w9"
# for p in [".nextflow", "nxf_logs", "nxf_work", "results"]:
#     shutil.rmtree(work_root/p, ignore_errors=True)
# shutil.rmtree(WORKSPACE_ROOT/"main/local_mock/mock/cache", ignore_errors=True)
    
agent.RunWorkflow(task)

In [ ]:
agent.CheckWorkflow(task.plan._key)

In [ ]:
# res_src = agent.GetResultSource(task, allow_globus=True)
# res_src

In [ ]:
# mover = Logistics()
# mover.QueueTransfer(
#     src = res_src,
#     dest = GlobusSource(
#         endpoint=chinook_ep,
#         path=f"/Metasmith/results.{task.plan._key}.xgdb",
#     ).AsSource(),
# )
# mover.ExecuteTransfers(wait_for_complete=False)

In [ ]:
# import mimetypes

# def istext(filename):
#     s=open(filename, encoding="latin1").read(512)
#     text_characters = "".join([chr(x) for x in range(32, 127)] + list("\n\r\t\b"))
#     translation_table = str.maketrans("", "", text_characters)
#     if not s:
#         # Empty files are considered text
#         return True
#     if "\0" in s:
#         # Files with null bytes are likely binary
#         return False
#     # Get the non-text characters (maps a character to itself then
#     # use the 'remove' option to get rid of the text characters.)
#     t = s.translate(translation_table)
#     # If more than 30% non-text characters, then
#     # this is considered a binary file
#     if float(len(t))/float(len(s)) > 0.30:
#         return False
#     return True

# # istext("/home/tony/workspace/tools/Metasmith/metasmith.sif")
# istext("/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz/nxf_work/dd/6d7e3eef979ec8010613bb376b63b6/container.diamond.oci.uri")

In [ ]:
# from metasmith.coms.containers import Runtime

# s = Runtime.APPTAINER.name
# Runtime[s], s